# Notebook 05: Process FANTOM5 CAGE-seq

Extract HEK293 untreated CAGE data from FANTOM5, annotate peaks with gene IDs
using the **FANTOM5 built-in peak annotation file**, and aggregate per gene.

**Method:** Use `fantom5_peaks_annotation.txt.gz` to assign peaks to genes
via the `short_description` column (format: `p1@GENENAME`). This matches
Jason's approach — peaks are assigned by TSS proximity, not gene-body overlap.

**Critical Test #3:** Compare annotate-first (correct) vs sum-first (old) approaches.

**Key details:**
- FANTOM5 expression matrix is already in TPM
- Only use CNhs11046 (HEK293/SLAM untreated)
- Annotate peaks with gene names from FANTOM5 annotation BEFORE summing
- Map gene names to Ensembl IDs for joining with Hubstenberger data
- Filter to protein-coding genes via Ensembl ID overlap

In [1]:
import pandas as pd
import numpy as np
import mygene
import gzip
import re
from pathlib import Path

RAW_DIR = Path("../data/raw")
CACHE_DIR = Path("../data/cache")
PROCESSED_DIR = Path("../data/processed")
REFERENCE_DIR = Path("../reference")

## 1. Load FANTOM5 Expression Matrix (HEK293 untreated only)

Extract only the CNhs11046 (untreated) column from the large expression matrix.

In [2]:
HEK293_LIBRARY_ID = "CNhs11046"  # HEK293/SLAM untreated ONLY

matrix_path = RAW_DIR / "fantom5_cage_tpm_matrix.osc.txt.gz"

# Read header to find the correct column index
print("Reading FANTOM5 expression matrix header...")
header_line = None
skip_rows = 0

with gzip.open(matrix_path, 'rt') as f:
    for line in f:
        if line.startswith('##'):
            skip_rows += 1
            continue
        header_line = line.strip()
        skip_rows += 1
        break

columns = header_line.split('\t')
hek_col_idx = None
for i, col in enumerate(columns):
    if HEK293_LIBRARY_ID in col:
        hek_col_idx = i
        print(f"Found HEK293 untreated column {i}: {col}")
        break

assert hek_col_idx is not None, f"Could not find {HEK293_LIBRARY_ID} in matrix header"

# Read only peak_id (col 0) and HEK293 untreated column
print(f"Reading 2 columns from expression matrix (skip {skip_rows} header rows)...")
cage_df = pd.read_csv(
    matrix_path, sep='\t', comment='#',
    usecols=[0, hek_col_idx], header=0,
    skiprows=skip_rows - 1
)
cage_df.columns = ["peak_id", "cage_tpm"]

# Drop non-peak rows (STAT rows)
cage_df = cage_df[cage_df["peak_id"].str.startswith("chr", na=False)].copy()

print(f"Loaded {len(cage_df):,} CAGE peaks")
print(f"Peaks with TPM > 0: {(cage_df['cage_tpm'] > 0).sum():,}")
print(f"Peaks with TPM = 0: {(cage_df['cage_tpm'] == 0).sum():,}")

Reading FANTOM5 expression matrix header...
Found HEK293 untreated column 1357: tpm.embryonic%20kidney%20cell%20line%3a%20HEK293%2fSLAM%20untreated.CNhs11046.10450-106F9
Reading 2 columns from expression matrix (skip 1832 header rows)...


Loaded 201,802 CAGE peaks
Peaks with TPM > 0: 65,248
Peaks with TPM = 0: 136,554


## 2. Load FANTOM5 Peak Annotation File

The annotation file maps each CAGE peak to its associated gene via TSS proximity.
Gene names are in the `short_description` column (format: `p1@GENENAME`).
This is how Jason assigned peaks to genes — not by gene-body overlap.

In [3]:
# Parse FANTOM5 peak annotation file
print("Parsing FANTOM5 peak annotation file...")
ann_rows = []
with gzip.open(RAW_DIR / "fantom5_peaks_annotation.txt.gz", 'rt') as f:
    header = None
    for line in f:
        line = line.strip()
        if line.startswith('##'):
            continue
        if header is None:
            header = line.split('\t')
            continue
        ann_rows.append(line.split('\t'))

ann = pd.DataFrame(ann_rows, columns=header)
print(f"Total peaks in annotation: {len(ann):,}")
print(f"Columns: {list(ann.columns)}")

# Extract gene name from short_description (format: p1@GENENAME or p@chr...)
ann["gene_name"] = ann["short_description"].str.extract(r'@([A-Za-z][A-Za-z0-9_.-]+)', expand=False)

has_gene = ann["gene_name"].notna()
print(f"\nPeaks with gene annotation: {has_gene.sum():,}")
print(f"Peaks without gene annotation: {(~has_gene).sum():,}")
print(f"Unique genes: {ann.loc[has_gene, 'gene_name'].nunique():,}")

Parsing FANTOM5 peak annotation file...


Total peaks in annotation: 201,802
Columns: ['00Annotation', 'short_description', 'description', 'association_with_transcript', 'entrezgene_id', 'hgnc_id', 'uniprot_id']

Peaks with gene annotation: 201,791
Peaks without gene annotation: 11
Unique genes: 26,277


## 3. Annotate Peaks with Gene Names, Then Map to Ensembl IDs

1. Join CAGE TPM data with annotation file on peak_id
2. Filter to peaks with a gene annotation
3. Map gene names to Ensembl IDs (needed for joining with Hubstenberger data)
4. Filter to protein-coding genes

In [4]:
# Join CAGE TPM with annotation on peak_id
cage_annotated = cage_df.merge(
    ann[["00Annotation", "gene_name"]],
    left_on="peak_id",
    right_on="00Annotation",
    how="inner"
)

print(f"Peaks matched to annotation: {len(cage_annotated):,} / {len(cage_df):,}")

# Keep only peaks that have a gene annotation
cage_annotated = cage_annotated[cage_annotated["gene_name"].notna()].copy()
print(f"Peaks with gene name: {len(cage_annotated):,}")
print(f"Unique genes: {cage_annotated['gene_name'].nunique():,}")

Peaks matched to annotation: 201,802 / 201,802
Peaks with gene name: 201,791
Unique genes: 26,277


In [5]:
# Map gene names to Ensembl IDs using mygene
unique_genes = cage_annotated["gene_name"].unique().tolist()
print(f"Querying mygene for {len(unique_genes):,} gene symbols...")

mg = mygene.MyGeneInfo()
results = mg.querymany(
    unique_genes, scopes="symbol",
    fields="ensembl.gene,type_of_gene",
    species="human", returnall=True
)

# Build gene_name -> ensembl_id mapping
name_to_ensembl = {}
name_to_biotype = {}
for r in results["out"]:
    if r.get("notfound", False):
        continue
    symbol = r["query"]
    ensembl = r.get("ensembl", {})
    # Handle multiple Ensembl entries
    if isinstance(ensembl, list):
        ensembl = ensembl[0]
    eid = ensembl.get("gene", None) if isinstance(ensembl, dict) else None
    if eid and symbol not in name_to_ensembl:
        name_to_ensembl[symbol] = eid
    biotype = r.get("type_of_gene", None)
    if biotype:
        name_to_biotype[symbol] = biotype

mapped = len(name_to_ensembl)
print(f"Mapped to Ensembl ID: {mapped:,} / {len(unique_genes):,}")
print(f"Protein-coding: {sum(1 for v in name_to_biotype.values() if v == 'protein-coding'):,}")

Input sequence provided is already in string format. No operation performed


Input sequence provided is already in string format. No operation performed


Querying mygene for 26,277 gene symbols...


616 input query terms found dup hits:	[('YWHAZP5', 2), ('RNU4-5P', 2), ('RPS10P18', 2), ('ALDOAP2', 2), ('HSD17B7P2', 2), ('CCNYL2', 2), (


9201 input query terms found no hit:	['chr10', 'ENST00000416191', 'BC029589', 'ENST00000436100', 'ENST00000444359', 'ENST00000537595', 'A


Mapped to Ensembl ID: 16,999 / 26,277
Protein-coding: 15,596


In [6]:
# Apply Ensembl ID and biotype to annotated peaks
cage_annotated["ensembl_id"] = cage_annotated["gene_name"].map(name_to_ensembl)
cage_annotated["gene_biotype"] = cage_annotated["gene_name"].map(name_to_biotype)

print(f"Peaks with Ensembl ID: {cage_annotated['ensembl_id'].notna().sum():,}")
print(f"Peaks without Ensembl ID: {cage_annotated['ensembl_id'].isna().sum():,}")

# Filter to protein-coding genes only
cage_pc = cage_annotated[cage_annotated["gene_biotype"] == "protein-coding"].copy()
print(f"\nProtein-coding peaks: {len(cage_pc):,}")
print(f"Protein-coding genes: {cage_pc['gene_name'].nunique():,}")

Peaks with Ensembl ID: 80,266
Peaks without Ensembl ID: 121,525

Protein-coding peaks: 77,763
Protein-coding genes: 15,596


## 4. Sum CAGE TPM Per Gene (Annotate-First Method)

In [7]:
# METHOD A (CORRECT): Peaks are already annotated via FANTOM5 annotation file.
# Now sum TPM per gene.
cage_per_gene = (
    cage_pc
    .groupby("ensembl_id")
    .agg(
        FANTOM_Total_CAGE_TPM=("cage_tpm", "sum"),
        FANTOM_n_peaks=("peak_id", "count")
    )
    .reset_index()
)

print(f"Genes with CAGE data: {len(cage_per_gene):,}")
print(f"\nCAGE TPM distribution:")
print(cage_per_gene["FANTOM_Total_CAGE_TPM"].describe())

Genes with CAGE data: 15,591

CAGE TPM distribution:
count    15591.000000
mean        27.775467
std        105.822128
min          0.000000
25%          0.000000
50%          6.013915
75%         23.433531
max       4617.027787
Name: FANTOM_Total_CAGE_TPM, dtype: float64


## 5. Critical Test: Annotate-First vs Sum-First

Compare Method A (annotate peaks with gene IDs from FANTOM5 annotation, then sum)
vs Method B (sum peaks by genomic position, then annotate).

With FANTOM5's TSS-based annotation (not gene-body overlap), the order difference
should be minimal since peaks have unique coordinates.

In [8]:
# METHOD B: Sum peaks by position first, then annotate
# Parse coordinates from peak_id
def parse_peak_coords(pid):
    m = re.match(r'(chr[\w]+):(\d+)\.\.(\d+),([+-])', str(pid))
    if m:
        return m.group(1), int(m.group(2)), int(m.group(3)), m.group(4)
    return None, None, None, None

cage_pc_coords = cage_pc.copy()
coords = cage_pc_coords["peak_id"].apply(parse_peak_coords)
cage_pc_coords["chr"] = [c[0] for c in coords]
cage_pc_coords["start"] = [c[1] for c in coords]
cage_pc_coords["end"] = [c[2] for c in coords]
cage_pc_coords["strand"] = [c[3] for c in coords]

# Sum by position first
cage_summed = (
    cage_pc_coords
    .groupby(["chr", "start", "end", "strand"])
    .agg(cage_tpm=("cage_tpm", "sum"), ensembl_id=("ensembl_id", "first"))
    .reset_index()
)

# Then aggregate per gene
cage_per_gene_B = (
    cage_summed
    .groupby("ensembl_id")
    .agg(
        FANTOM_Total_CAGE_TPM=("cage_tpm", "sum"),
        FANTOM_n_peaks=("cage_tpm", "count")
    )
    .reset_index()
)

print(f"Method A (annotate-first): {len(cage_per_gene):,} genes")
print(f"Method B (sum-first):      {len(cage_per_gene_B):,} genes")

# Compare
comp = cage_per_gene.merge(cage_per_gene_B, on="ensembl_id", how="outer", suffixes=("_A", "_B"))
both = comp.dropna(subset=["FANTOM_Total_CAGE_TPM_A", "FANTOM_Total_CAGE_TPM_B"])
diff = (both["FANTOM_Total_CAGE_TPM_A"] - both["FANTOM_Total_CAGE_TPM_B"]).abs()

print(f"\nGenes in both: {len(both):,}")
print(f"TPM exact matches: {(diff < 1e-6).sum():,} / {len(both):,}")
print(f"Max diff: {diff.max():.6f}")

Method A (annotate-first): 15,591 genes
Method B (sum-first):      15,591 genes

Genes in both: 15,591
TPM exact matches: 15,591 / 15,591
Max diff: 0.000000


## 6. Validation Against Jason's Reference

Compare our per-gene CAGE TPM against Jason's `FANTOM_Total_CAGE_TPM` column.

In [9]:
from scipy import stats as sp_stats

ref = pd.read_csv(REFERENCE_DIR / "12544_Hub_CAGE_MERGE_with_CapIndex.csv")

val = cage_per_gene.merge(
    ref[["ensembl_id", "FANTOM_Total_CAGE_TPM", "FANTOM_n_peaks"]],
    on="ensembl_id", how="inner", suffixes=("_ours", "_ref")
)

print(f"Genes compared: {len(val):,} / {len(ref):,} reference genes")

# Genes in reference but not in our output
missing = set(ref["ensembl_id"]) - set(cage_per_gene["ensembl_id"])
ref_missing = ref[ref["ensembl_id"].isin(missing)]
zero_cage = (ref_missing["FANTOM_Total_CAGE_TPM"] == 0).sum()
print(f"Reference genes missing from our output: {len(missing):,}")
print(f"  Of those, with CAGE TPM = 0 in ref: {zero_cage:,}")

# Compare CAGE TPM
mask = (val["FANTOM_Total_CAGE_TPM_ours"] > 0) | (val["FANTOM_Total_CAGE_TPM_ref"] > 0)
r, _ = sp_stats.pearsonr(val.loc[mask, "FANTOM_Total_CAGE_TPM_ours"],
                         val.loc[mask, "FANTOM_Total_CAGE_TPM_ref"])
d = (val["FANTOM_Total_CAGE_TPM_ours"] - val["FANTOM_Total_CAGE_TPM_ref"]).abs()
close = np.allclose(val["FANTOM_Total_CAGE_TPM_ours"], val["FANTOM_Total_CAGE_TPM_ref"],
                    rtol=1e-5, atol=1e-8)
exact = (d < 1e-6).sum()

print(f"\nFANTOM_Total_CAGE_TPM:")
print(f"  Pearson r: {r:.10f}")
print(f"  Max diff: {d.max():.6f}")
print(f"  Mean diff: {d.mean():.6f}")
print(f"  Exact (<1e-6): {exact:,} / {len(val):,}")
print(f"  allclose(rtol=1e-5): {'PASS' if close else 'FAIL'}")

# Peak count comparison
peak_exact = (val["FANTOM_n_peaks_ours"] == val["FANTOM_n_peaks_ref"]).sum()
print(f"\nFANTOM_n_peaks:")
print(f"  Exact matches: {peak_exact:,} / {len(val):,}")

# Show worst discrepancies
if d.max() > 0.01:
    print(f"\nWorst CAGE TPM discrepancies:")
    worst = val.loc[d.nlargest(10).index]
    for _, row in worst.iterrows():
        gn = ref[ref["ensembl_id"]==row["ensembl_id"]]["Associated Gene Name"].values[0]
        print(f"  {gn:<15} ours={row['FANTOM_Total_CAGE_TPM_ours']:>10.2f}  ref={row['FANTOM_Total_CAGE_TPM_ref']:>10.2f}  "
              f"peaks: ours={int(row['FANTOM_n_peaks_ours'])}, ref={int(row['FANTOM_n_peaks_ref'])}")

Genes compared: 12,315 / 12,544 reference genes
Reference genes missing from our output: 229
  Of those, with CAGE TPM = 0 in ref: 46

FANTOM_Total_CAGE_TPM:
  Pearson r: 0.9965701965
  Max diff: 735.564030
  Mean diff: 0.282311
  Exact (<1e-6): 12,232 / 12,315
  allclose(rtol=1e-5): FAIL

FANTOM_n_peaks:
  Exact matches: 12,209 / 12,315

Worst CAGE TPM discrepancies:
  CALM1           ours=    741.16  ref=      5.60  peaks: ours=22, ref=19
  NME1            ours=    278.92  ref=      0.00  peaks: ours=2, ref=1
  MRPL12          ours=    263.47  ref=      1.66  peaks: ours=7, ref=2
  ARHGAP4         ours=    214.43  ref=      0.00  peaks: ours=4, ref=3
  DAGLB           ours=    202.09  ref=     11.20  peaks: ours=3, ref=2
  RCC1            ours=    144.23  ref=     12.24  peaks: ours=4, ref=3
  NDUFA13         ours=    126.08  ref=      0.00  peaks: ours=3, ref=2
  PPP2CA          ours=    130.23  ref=      8.92  peaks: ours=5, ref=4
  SYCP2L          ours=    114.68  ref=      3.42  

## 7. Save Output

In [10]:
output_path = PROCESSED_DIR / "05_fantom5_cage_per_gene.csv"
cage_per_gene.to_csv(output_path, index=False)
print(f"Saved {len(cage_per_gene):,} genes with CAGE data to {output_path}")

Saved 15,591 genes with CAGE data to ../data/processed/05_fantom5_cage_per_gene.csv
